# Building Interactive Reports with Pandas and Matplotlib

This notebook builds on the foundations from the previous two tutorials. Here we explore generating multiple plots programmatically using loops, modifying DataFrames by adding and removing data, using pandas' built-in plotting, creating interactive elements with ipywidgets including data entry forms, and assembling saved figures into a cohesive report.

The notebook includes compatibility guidance for both standard Jupyter environments (including edupyter) and Google Colab. Where the two differ, you will find comments indicating which approach to use.

## Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    print("Note: Files created here are temporary unless saved to Google Drive")
except ImportError:
    IN_COLAB = False
    print("Running in standard Jupyter environment")

%matplotlib inline

# Create a directory for saved figures
os.makedirs('report_figures', exist_ok=True)

In [ ]:
# Colab-specific: Mount Google Drive for persistent file storage
# Uncomment these lines when running in Colab if you want files to persist

# if IN_COLAB:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     # Then save files to /content/drive/MyDrive/your_folder/

## Working Dataset

Let's create a dataset representing annual environmental indicators for several countries.

In [ ]:
np.random.seed(42)

countries = ['Germany', 'France', 'UK', 'Spain', 'Italy', 'Poland', 'Netherlands', 'Sweden']
years = list(range(2015, 2024))

data_rows = []

base_renewable = {'Germany': 30, 'France': 20, 'UK': 25, 'Spain': 35, 
                  'Italy': 35, 'Poland': 12, 'Netherlands': 12, 'Sweden': 55}
base_emissions = {'Germany': 800, 'France': 300, 'UK': 350, 'Spain': 250,
                  'Italy': 320, 'Poland': 310, 'Netherlands': 150, 'Sweden': 40}

for country in countries:
    for i, year in enumerate(years):
        renewable = base_renewable[country] + i * 2.5 + np.random.normal(0, 2)
        renewable = min(renewable, 85)
        
        emissions = base_emissions[country] - i * 8 + np.random.normal(0, 15)
        emissions = max(emissions, 30)
        
        consumption = 200 + len(country) * 20 + np.random.normal(0, 30)
        
        data_rows.append({
            'country': country,
            'year': year,
            'renewable_percent': round(renewable, 1),
            'co2_emissions_mt': round(emissions, 1),
            'energy_consumption_twh': round(consumption, 1)
        })

df = pd.DataFrame(data_rows)
print(f"Dataset shape: {df.shape}")
df.head(10)

## Modifying DataFrames: Adding and Removing Data

### Adding Columns

In [ ]:
# Calculated column
df['emissions_per_twh'] = df['co2_emissions_mt'] / df['energy_consumption_twh']
df.head()

In [ ]:
# Conditional column with np.where()
df['high_renewable'] = np.where(df['renewable_percent'] > 40, 'Yes', 'No')
df[df['year'] == 2023][['country', 'renewable_percent', 'high_renewable']]

In [ ]:
# Multiple conditions with np.select()
conditions = [
    df['renewable_percent'] >= 50,
    df['renewable_percent'] >= 30,
    df['renewable_percent'] >= 15
]
labels = ['Leader', 'Progressing', 'Developing']

df['renewable_category'] = np.select(conditions, labels, default='Lagging')
df['renewable_category'].value_counts()

### Removing Columns

In [ ]:
df = df.drop(columns=['high_renewable'])
print("Columns:", list(df.columns))

### Adding and Removing Rows

In [ ]:
# Add rows with pd.concat()
new_data = pd.DataFrame([{
    'country': 'Denmark', 'year': 2023, 'renewable_percent': 84.0,
    'co2_emissions_mt': 25.0, 'energy_consumption_twh': 32.5,
    'emissions_per_twh': 0.77, 'renewable_category': 'Leader'
}])

df = pd.concat([df, new_data], ignore_index=True)
df[df['country'] == 'Denmark']

In [ ]:
# Remove rows by condition
print(f"Before: {len(df)} rows")
df_recent = df[df['year'] >= 2020].copy()
print(f"After filtering to 2020+: {len(df_recent)} rows")

## Pandas Built-in Plotting

In [ ]:
germany = df[df['country'] == 'Germany'].sort_values('year')

germany.plot(x='year', y='renewable_percent', kind='line', 
             figsize=(10, 5), title='Germany Renewable Energy %',
             color='green', marker='o')
plt.ylabel('Renewable %')
plt.show()

In [ ]:
latest = df[df['year'] == 2023].set_index('country')
latest['renewable_percent'].plot(kind='bar', figsize=(10, 5),
                                  color='teal', edgecolor='white',
                                  title='Renewable Energy % by Country (2023)')
plt.ylabel('Renewable %')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Creating Multiple Plots with Loops

In [ ]:
countries_in_data = df['country'].unique()

# Grid of subplots
n_countries = len(countries_in_data)
n_cols = 3
n_rows = (n_countries + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows), sharey=True)
axes = axes.flatten()

for i, country in enumerate(countries_in_data):
    country_data = df[df['country'] == country].sort_values('year')
    axes[i].plot(country_data['year'], country_data['renewable_percent'],
                 marker='o', linewidth=2, color='teal')
    axes[i].set_title(country)
    axes[i].set_xlabel('Year')
    if i % n_cols == 0:
        axes[i].set_ylabel('Renewable %')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Renewable Energy Trends by Country', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Save figures in a loop
saved_files = []

for country in countries_in_data:
    country_data = df[df['country'] == country].sort_values('year')
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(country_data['year'], country_data['renewable_percent'],
            marker='o', linewidth=2, color='teal')
    ax.fill_between(country_data['year'], country_data['renewable_percent'],
                    alpha=0.3, color='teal')
    ax.set_xlabel('Year')
    ax.set_ylabel('Renewable Energy %')
    ax.set_title(f'{country}: Renewable Energy Trend')
    
    safe_name = country.lower().replace(' ', '_')
    filename = f'report_figures/{safe_name}_renewable.png'
    fig.savefig(filename, dpi=150, bbox_inches='tight')
    saved_files.append(filename)
    plt.close(fig)

print(f"Saved {len(saved_files)} figures")

## Interactive Visualisation with ipywidgets

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Colab may need this for full widget support
if IN_COLAB:
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
        print("Colab widget manager enabled")
    except:
        print("Basic widget support")

In [ ]:
from ipywidgets import interact

def plot_country_trend(country):
    country_data = df[df['country'] == country].sort_values('year')
    
    plt.figure(figsize=(10, 5))
    plt.plot(country_data['year'], country_data['renewable_percent'],
             marker='o', linewidth=2, color='teal')
    plt.fill_between(country_data['year'], country_data['renewable_percent'],
                     alpha=0.3, color='teal')
    plt.xlabel('Year')
    plt.ylabel('Renewable Energy %')
    plt.title(f'{country}: Renewable Energy Trend')
    plt.ylim(0, 90)
    plt.tight_layout()
    plt.show()

interact(plot_country_trend, country=list(countries_in_data));

## Data Entry Forms

Interactive forms let users add data without editing code.

In [ ]:
# Data entry form
entries = []

station_input = widgets.Text(description='Station:', placeholder='e.g., Dublin Central')
temp_input = widgets.FloatText(description='Temp (C):', value=10.0)
aqi_input = widgets.IntSlider(description='AQI:', min=0, max=200, value=50)
rainfall_input = widgets.FloatText(description='Rainfall (mm):', value=0.0)

add_button = widgets.Button(description='Add Entry', button_style='success', icon='plus')
clear_button = widgets.Button(description='Clear All', button_style='danger', icon='trash')
save_button = widgets.Button(description='Save to CSV', button_style='info', icon='save')

output = widgets.Output()

def display_entries():
    with output:
        clear_output()
        if entries:
            display(pd.DataFrame(entries))
            print(f"\nTotal entries: {len(entries)}")
        else:
            print("No entries yet.")

def add_entry(b):
    if station_input.value.strip():
        entries.append({
            'station': station_input.value.strip(),
            'temperature_c': temp_input.value,
            'aqi': aqi_input.value,
            'rainfall_mm': rainfall_input.value
        })
        station_input.value = ''
        temp_input.value = 10.0
        aqi_input.value = 50
        rainfall_input.value = 0.0
    display_entries()

def clear_entries(b):
    entries.clear()
    display_entries()

def save_entries(b):
    with output:
        if entries:
            filename = 'environmental_readings.csv'
            pd.DataFrame(entries).to_csv(filename, index=False)
            print(f"\nSaved to {filename}")
            if IN_COLAB:
                from google.colab import files
                files.download(filename)
        else:
            print("\nNo entries to save.")

add_button.on_click(add_entry)
clear_button.on_click(clear_entries)
save_button.on_click(save_entries)

form = widgets.VBox([
    widgets.HTML('<h3>Environmental Data Entry</h3>'),
    station_input, temp_input, aqi_input, rainfall_input,
    widgets.HBox([add_button, clear_button, save_button]),
    widgets.HTML('<hr>'),
    output
])

display(form)
display_entries()

## Practice Exercises

**Exercise 1:** Create a data entry form for collecting book information (title, author, year, rating). Include a button to save the entries to a CSV file.

In [ ]:
# Your code here


**Exercise 2:** Write a loop that creates and saves a separate bar chart for each year in the dataset, showing renewable_percent by country for that year.

In [ ]:
# Your code here


**Exercise 3:** Create an interactive widget with two dropdowns (country1, country2) that displays both countries' renewable trends on the same chart.

In [ ]:
# Your code here


## Colab Compatibility Summary

Most code works identically in standard Jupyter and Google Colab. Key differences:

**File persistence:** Colab's filesystem is temporary. Mount Google Drive for persistent storage, or use `files.download()` to save to your computer.

**Widgets:** Basic ipywidgets work in both. For full support in Colab, include `output.enable_custom_widget_manager()`.

**Saving files:** The save buttons automatically offer downloads in Colab.

## Bibliography

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. https://wesmckinney.com/book/

ipywidgets documentation. https://ipywidgets.readthedocs.io/

Matplotlib documentation. https://matplotlib.org/stable/tutorials/

Google Colab documentation. https://colab.research.google.com/notebooks/